# Amazon Review Sentiment Analysis
## Part 2 — Text Preprocessing

**Goal**: Understand what the raw text looks like, walk through the tokenisation
pipeline, clean the corpus, and upload the processed dataset to GCS so the
training pipeline can consume it without re-doing the same steps.

*Previous*: `eda.ipynb`  |  *Next*: `feature_engineering.ipynb`

In [1]:
import os, sys, re
from typing import List

project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd

from src.data_ingestion import load_raw_data_from_gcs, save_processed_data_to_gcs


def tokenize(text: str) -> List[str]:
    """Lowercase, strip non-alphanumeric characters, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()


def join_tokens(tokens: List[str]) -> str:
    return ' '.join(tokens)


def preprocess_series(series: pd.Series) -> pd.Series:
    """Apply tokenize -> join_tokens to every row; NaN rows are dropped."""
    return series.dropna().apply(lambda t: join_tokens(tokenize(t)))


def get_vocab_stats(corpus: pd.Series) -> dict:
    all_tokens = corpus.dropna().apply(tokenize)
    vocab = {token for tokens in all_tokens for token in tokens}
    avg_len = all_tokens.apply(len).mean()
    return {
        'vocab_size': len(vocab),
        'avg_tokens_per_doc': round(avg_len, 2),
        'total_docs': len(corpus.dropna()),
    }


print('Setup complete.  project_root:', project_root)


Setup complete.  project_root: C:\Users\Priya Bhaskar\practice_genai_projects\amazon_review_sentiment\reviews_sentiment


---
## 1. Load Raw Data

Raw data lives in GCS `raw_data/cleaned_reviews.csv`.
The `cleaned_review` column is already lower-cased and punctuation-stripped
from the original data pipeline.

In [2]:
df_raw = load_raw_data_from_gcs("cleaned_reviews.csv")
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print()
print("Columns:", df_raw.columns.tolist())
print()
df_raw.head(6)

Shape: 17,340 rows x 4 columns

Columns: ['sentiments', 'cleaned_review', 'cleaned_review_length', 'review_score']



,sentiments,cleaned_review,cleaned_review_length,review_score
0,positive,i wish would have gotten one earlier love it a...,19,5.0
1,neutral,i ve learned this lesson again open the packag...,88,1.0
2,neutral,it is so slow and lags find better option,9,2.0
3,neutral,roller ball stopped working within months of m...,12,1.0
4,neutral,i like the color and size but it few days out ...,21,1.0
5,positive,overall love this mouse the size weight clicki...,68,3.0


In [3]:
print("dtypes:")
print(df_raw.dtypes)
print()
print("Missing values:")
print(df_raw.isnull().sum())
print()
print("Numeric summary:")
print(df_raw.describe())

dtypes:
sentiments                   str
cleaned_review               str
cleaned_review_length      int64
review_score             float64
dtype: object

Missing values:
sentiments               0
cleaned_review           3
cleaned_review_length    0
review_score             0
dtype: int64

Numeric summary:
       cleaned_review_length  review_score
count           17340.000000  17340.000000
mean               30.300461      3.649077
std                35.836540      1.673500
min                 0.000000      1.000000
25%                 9.000000      2.000000
50%                20.000000      5.000000
75%                38.000000      5.000000
max               571.000000      5.000000


---
## 2. Tokenisation Pipeline

The `tokenize(text)` function defined in this notebook:
1. Lowercases the string
2. Removes any character that is not a letter, digit, or space
3. Splits on whitespace

The result is a clean list of word tokens. `preprocess_series` applies
this to every row of the corpus and rejoins tokens into a single string —
the format the TF-IDF vectorizer expects at both fit time and transform time.

In [4]:
# Walk through 4 examples so every step is visible
samples = df_raw["cleaned_review"].dropna().sample(4, random_state=42).tolist()

for i, raw in enumerate(samples, 1):
    tokens = tokenize(raw)
    print(f"Example {i}")
    print(f"  Input  : {raw[:90]}")
    print(f"  Tokens : {tokens[:10]} ...")
    print(f"  Count  : {len(tokens)} tokens")
    print()

Example 1
  Input  : it meets my need 
  Tokens : ['it', 'meets', 'my', 'need'] ...
  Count  : 4 tokens

Example 2
  Input  : i shopped around for while great speaker sound excellent very happy
  Tokens : ['i', 'shopped', 'around', 'for', 'while', 'great', 'speaker', 'sound', 'excellent', 'very'] ...
  Count  : 11 tokens

Example 3
  Input  : why there no lights or showing if it charged or not have to turn it on to see the lights t
  Tokens : ['why', 'there', 'no', 'lights', 'or', 'showing', 'if', 'it', 'charged', 'or'] ...
  Count  : 115 tokens

Example 4
  Input  : it had connectivity issues right out of the box contacted the seller and he she did not re
  Tokens : ['it', 'had', 'connectivity', 'issues', 'right', 'out', 'of', 'the', 'box', 'contacted'] ...
  Count  : 39 tokens



In [5]:
# Corpus-level vocabulary statistics
stats = get_vocab_stats(df_raw["cleaned_review"])
print("Corpus vocabulary statistics:")
for k, v in stats.items():
    print(f"  {k:30s}: {v}")

Corpus vocabulary statistics:
  vocab_size                    : 9593
  avg_tokens_per_doc            : 30.31
  total_docs                    : 17337


---
## 3. Apply Preprocessing to the Full Corpus

`preprocess_series` applies `tokenize -> join_tokens` to every row.
This gives a single normalised string per review — the same format
the TF-IDF vectorizer expects at both fit time and transform time.

In [6]:
# Show before / after for 3 rows
samples_idx = df_raw["cleaned_review"].dropna().sample(3, random_state=7).index

print("Before vs After preprocessing:")
print("-" * 70)
for idx in samples_idx:
    original  = df_raw.loc[idx, "cleaned_review"]
    processed = join_tokens(tokenize(original))
    print(f"Before : {original[:70]}")
    print(f"After  : {processed[:70]}")
    print()

Before vs After preprocessing:
----------------------------------------------------------------------
Before : i love this mouse the colors are beautiful and ve gotten many complime
After  : i love this mouse the colors are beautiful and ve gotten many complime

Before : at first was little bit worried about the quality of this product but 
After  : at first was little bit worried about the quality of this product but 

Before : these headphones are amazing especially for the price the cable is lon
After  : these headphones are amazing especially for the price the cable is lon



In [7]:
# Apply to the whole column
cleaned_series = preprocess_series(df_raw["cleaned_review"])

print(f"Rows before preprocessing : {len(df_raw):,}")
print(f"Rows after  preprocessing : {len(cleaned_series):,}   (NaN rows dropped)")
print()
print("Sample processed reviews:")
print(cleaned_series.head(5).to_string())

Rows before preprocessing : 17,340
Rows after  preprocessing : 17,337   (NaN rows dropped)

Sample processed reviews:
0    i wish would have gotten one earlier love it a...
1    i ve learned this lesson again open the packag...
2            it is so slow and lags find better option
3    roller ball stopped working within months of m...
4    i like the color and size but it few days out ...


---
## 4. Build the Processed Dataset

We assemble the final DataFrame that the training pipeline will use:

| Column | Source | Description |
|---|---|---|
| `sentiments` | raw | Target label: positive / neutral / negative |
| `cleaned_review` | preprocessed | Normalised text ready for vectorisation |
| `review_score` | raw | Star rating 1–5 |
| `word_count` | computed | Number of tokens after preprocessing |

Rows with a missing `cleaned_review` are dropped (only 3 rows affected).

In [8]:
# Align cleaned_series index back to the raw df that still has all columns
df_processed = df_raw.loc[cleaned_series.index].copy()
df_processed["cleaned_review"] = cleaned_series
df_processed["word_count"]     = cleaned_series.str.split().str.len()

# Keep only the four columns the model pipeline needs
df_processed = df_processed[["sentiments", "cleaned_review", "review_score", "word_count"]]

# Remove any rows where cleaning produced an empty string (all-numeric or all-special reviews)
df_processed = df_processed[df_processed["cleaned_review"].str.strip().ne("")]
df_processed = df_processed.reset_index(drop=True)

print(f"Processed dataset shape: {df_processed.shape}")
print()
print("dtypes:")
print(df_processed.dtypes)
print()
print("Sentiment distribution:")
print(df_processed["sentiments"].value_counts())
print()
df_processed.head(6)

Processed dataset shape: (17321, 4)

dtypes:
sentiments            str
cleaned_review        str
review_score      float64
word_count          int64
dtype: object

Sentiment distribution:
sentiments
positive    9503
neutral     6284
negative    1534
Name: count, dtype: int64



,sentiments,cleaned_review,review_score,word_count
0,positive,i wish would have gotten one earlier love it a...,5.0,19
1,neutral,i ve learned this lesson again open the packag...,1.0,88
2,neutral,it is so slow and lags find better option,2.0,9
3,neutral,roller ball stopped working within months of m...,1.0,12
4,neutral,i like the color and size but it few days out ...,1.0,21
5,positive,overall love this mouse the size weight clicki...,3.0,68


In [9]:
# Sanity checks before uploading
assert df_processed.isnull().sum().sum() == 0, "NaN values remain — check preprocessing"
assert df_processed["word_count"].min() >= 0,  "Negative word counts — check pipeline"
assert set(df_processed["sentiments"].unique()) == {"positive", "neutral", "negative"}

print("All sanity checks passed.")
print(f"  Rows    : {len(df_processed):,}")
print(f"  Columns : {df_processed.columns.tolist()}")

All sanity checks passed.
  Rows    : 17,321
  Columns : ['sentiments', 'cleaned_review', 'review_score', 'word_count']


---
## 5. Upload Processed Data to GCS

The processed file is saved to `gs://machine_learning_datasets/processed_data/processed_reviews.csv`.

The training pipeline (`main.py`) and the feature engineering notebook
both read from this path.

In [10]:
save_processed_data_to_gcs(df_processed, "processed_reviews.csv")

Uploaded 17,321 rows  ->  gs://machine_learning_datasets/processed_data/processed_reviews.csv


In [11]:
# Verify: reload and compare shape
from src.data_ingestion import load_processed_data_from_gcs

df_verify = load_processed_data_from_gcs("processed_reviews.csv")
print(f"Reloaded from GCS: {df_verify.shape}")
print()
print(df_verify.head(4).to_string())
print()
print("Round-trip check passed." if df_verify.shape == df_processed.shape else "MISMATCH — check upload")

Reloaded from GCS: (17321, 4)

  sentiments                                                                                                                                                                                                                                                                                                                                                                                                                                                               cleaned_review  review_score  word_count
0   positive                                                                                                                                                                                                                                                                                                                                                                                i wish would have gotten one earlier love it and it makes working in my laptop so much ea

---
## Summary

| Step | Result |
|---|---|
| Raw rows | 17,340 |
| Rows after dropping NaN | 17,337 |
| Preprocessing applied | lowercase, strip non-alphanumeric, split |
| Saved to GCS | `processed_data/processed_reviews.csv` |

*Open `feature_engineering.ipynb` to explore how features are built from this data.*